# Teacher smoke-test notebook

Walks through every teacher feature. Use this before deploying to catch regressions.
See `docs/smoke-test.md` for the full pre-deploy checklist (web steps included).

**Prereqs:**
- Backend running locally (e.g. `docker-compose up`).
- `pip install -e ./jupyter-integration` so `%load_ext cadence` works.
- A teacher account already created via the web at `http://localhost:3000/signup`.
  (You can sign up first, then come back here.)

In [ ]:
%load_ext cadence

## 1. Sign in

`%cadence_login` prompts for your password (never echoes it). The JWT is cached at
`~/.cadence/credentials.yaml` with 0600 perms so subsequent kernels don't ask again.

In [ ]:
%cadence_login --username smoke_teacher

In [ ]:
%cadence_whoami

## 2. Create a quick lesson

Because you're logged in, Cadence assumes you accepted the Terms at web
signup — no separate attestation step. If you ever use lesson commands
*without* logging in (token-only flow), the first create call will show
an inline prompt linking to `/terms` and `/privacy`; type `yes` to accept
and the command continues in the same cell.

Lessons are token-only — anyone with the teacher token can manage them.
They default to a 7-day session retention. The output card shows the join
code (give to students) and the dashboard URL (bookmark it).

In [ ]:
%cadence_create_lesson "Smoke quick lesson"

### Register a few checkpoints

The `--expected` value is JSON, embedded inside single quotes so the shell
doesn't interpret it. Examples below cover the four auto-checkable
comparators — `exact`, `numeric` (with tolerance), `set` (order-insensitive),
and `manual` (reflection-style, no auto-check). The two ways to register:

- **One at a time** with `%cadence_register <id> --comparator X --expected '<json>'`
  (good for ad-hoc additions).
- **Many at once** with `%%cadence_register_yaml` (good for the bulk of a
  lesson, especially when you have multi-line solution code).

In [ ]:
%cadence_register hello --comparator exact --expected '"Hello, World!"' \
    --hint "Watch the punctuation."

In [ ]:
%cadence_register mean --comparator numeric \
    --expected '{"value": 49.5, "tolerance": 0.001}' \
    --hint "Average of 0..99."

In [ ]:
%cadence_register row-sums --comparator set \
    --expected '{"value": [6, 22, 38]}' \
    --hint "axis=1"

In [ ]:
%cadence_register reflect --comparator manual \
    --hint "What does the peak shape tell you?"

### Bulk register with YAML (multi-line solution code)

For the headline checkpoint we want a hint, a worked solution that the
student can request after a few wrong tries, AND we want to accept code
submissions. The YAML form keeps multi-line code readable. Field names
mirror the flags on `%cadence_register` (snake_case).

In [ ]:
%%cadence_register_yaml
- id: higgs-peak
  comparator: exact
  expected: 125
  hint: Integer centre of the 1-GeV bin with the most events.
  reveal_after: 3
  solution_value: "125"
  solution_code: |
    import numpy as np
    bin_edges = np.arange(100, 151)
    counts, _ = np.histogram(m_gg, bins=bin_edges)
    int(bin_edges[np.argmax(counts)])
  allow_submissions: true

### Self-test: verify the expected answers parse and resolve

Submits the teacher's own expected value for every auto-checked checkpoint.
Catches typos in `--expected` or tolerance errors before any student sees them.
Manual / regex checkpoints are skipped.

In [ ]:
%cadence_self_test

## 3. Create a course (requires login)

Courses are owned by a teacher account — try this without `%cadence_login`
and you'll get a clear "sign in required" message. With the JWT cached you
should see the green confirmation card.

Once created, the course **appears automatically in your library at
`/teacher/library`** — no need to paste any token. The library page reads
`/courses/mine` and shows everything you own.

In [ ]:
%cadence_create_course "Smoke course"

In [ ]:
%cadence_add_notebook "Week 1 — Warmups"

## 4. Soft warning on long retention

The next cell asks for 300-day retention without confirmation — you should
see an amber warning explaining why long per-student retention is a bad
default, with instructions to re-run with `--yes-long-retention`.

In [ ]:
%cadence_create_course "Long-retention course" --retention-days 300

In [ ]:
%cadence_create_course "Long-retention course (confirmed)" --retention-days 300 --yes-long-retention

## 5. Rotate the teacher token

Useful if a token leaks. The old token is dead immediately; the local cache
and the dashboard URL get updated. `--also-join-code` regenerates the join
code too (disconnects current students — only for hard revocation).

In [ ]:
%cadence_rotate_token

## 6. Sign out (optional)

Clears the cached JWT. After this, course-creation commands will block
until you log back in. Lesson commands keep working (they use the token in
`~/.cadence/lessons.yaml`, not the JWT).

In [ ]:
%cadence_logout

## Done

**Save the join code from `%cadence_create_lesson` above** — paste it into
`student-example.ipynb` to walk through the student-side smoke test next.

Verify on the dashboard URL (also printed above) that the lesson appears,
checkpoints are registered, and the student roster updates in real time
once the student notebook is running.